# Association of Variables with Time Headway — Non-parametric Analysis

**Outcome:** `Time_Headway` (continuous, seconds)

For each candidate variable, the test is chosen by its measurement scale:

| Variable type | Test | Effect size reported |
|---|---|---|
| Continuous | Spearman rank correlation | Spearman \(\rho\) |
| Binary categorical (2 groups) | Mann–Whitney U | Rank-biserial \(r\) |
| Multi-level categorical (>2 groups) | Kruskal–Wallis H | Epsilon-squared \(\varepsilon^2\) |

Because 10 tests are run simultaneously, raw p-values are also reported with a
Benjamini–Hochberg (FDR) correction. Output table is written to the `Tables` folder.

In [1]:
# --- Cell 1: Imports and paths ---
import os
import numpy as np
import pandas as pd
from scipy import stats

BASE      = r"D:\Headway"
DATA_PATH = os.path.join(BASE, "data2.xlsx")
GRAPHICS  = os.path.join(BASE, "Graphics")
TABLES    = os.path.join(BASE, "Tables")
os.makedirs(GRAPHICS, exist_ok=True)
os.makedirs(TABLES,   exist_ok=True)

ALPHA = 0.05
OUTCOME = "Time_Headway"

In [2]:
# --- Cell 2: Load data and quick check ---
df = pd.read_excel(DATA_PATH)
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nMissing values:\n", df.isna().sum())
print("\nOutcome summary:\n", df[OUTCOME].describe())

Shape: (898, 11)

Columns: ['V_Subject', 'V_Leading_Class', 'Pair', 'Time_Headway', 'Target_Speed_km/hr', 'Leading_Speed_km/hr', 'Speed_Difference', 'Off_centeredness', 'Occupancy', 'Flow_pcu/hr', 'Site']

Missing values:
 V_Subject              0
V_Leading_Class        0
Pair                   0
Time_Headway           0
Target_Speed_km/hr     0
Leading_Speed_km/hr    0
Speed_Difference       0
Off_centeredness       0
Occupancy              0
Flow_pcu/hr            0
Site                   0
dtype: int64

Outcome summary:
 count    898.000000
mean       2.227394
std        0.988410
min        0.533333
25%        1.433333
50%        2.100000
75%        2.933333
max        4.966667
Name: Time_Headway, dtype: float64


In [3]:
# --- Cell 3: Declare each predictor's measurement scale ---
# Explicit mapping keeps the test choice transparent and correct
# (bool columns are BINARY categorical, not continuous).
VARIABLE_SCALE = {
    "V_Subject":           "binary",       # BTW / PR
    "V_Leading_Class":     "multi",        # 4 classes
    "Pair":                "multi",        # 8 pairs
    "Target_Speed_km/hr":  "continuous",
    "Leading_Speed_km/hr": "continuous",
    "Speed_Difference":    "continuous",
    "Off_centeredness":    "binary",       # True / False
    "Occupancy":           "binary",       # True / False
    "Flow_pcu/hr":         "continuous",
    "Site":                "binary",       # Tikatuli / Shahjahanpur
}
# safety: only keep predictors that exist and are not the outcome
VARIABLE_SCALE = {k: v for k, v in VARIABLE_SCALE.items()
                  if k in df.columns and k != OUTCOME}
VARIABLE_SCALE

{'V_Subject': 'binary',
 'V_Leading_Class': 'multi',
 'Pair': 'multi',
 'Target_Speed_km/hr': 'continuous',
 'Leading_Speed_km/hr': 'continuous',
 'Speed_Difference': 'continuous',
 'Off_centeredness': 'binary',
 'Occupancy': 'binary',
 'Flow_pcu/hr': 'continuous',
 'Site': 'binary'}

In [4]:
# --- Cell 4: Effect-size helpers and interpretation thresholds ---
def rank_biserial(u_stat, n1, n2):
    """Rank-biserial correlation for Mann-Whitney U (magnitude 0-1)."""
    return 1.0 - (2.0 * u_stat) / (n1 * n2)

def epsilon_squared(h_stat, n):
    """Epsilon-squared effect size for Kruskal-Wallis (0-1).
    eps^2 = H / (n - 1)  [equivalent to H(n+1)/(n^2-1)]."""
    return h_stat / (n - 1)

def interpret_r(r):
    r = abs(r)
    if r < 0.10: return "negligible"
    if r < 0.30: return "weak"
    if r < 0.50: return "moderate"
    return "strong"

def interpret_eps2(e):
    # eta-squared style conventions
    if e < 0.01: return "negligible"
    if e < 0.06: return "small"
    if e < 0.14: return "moderate"
    return "large"

def group_medians(sub_df, cat_col):
    med = sub_df.groupby(cat_col)[OUTCOME].median().round(3)
    return "; ".join(f"{k}={v}" for k, v in med.items())

In [5]:
# --- Cell 5: Run the appropriate non-parametric test per variable ---
rows = []
for var, scale in VARIABLE_SCALE.items():
    sub = df[[var, OUTCOME]].dropna()
    n = len(sub)

    if scale == "continuous":
        rho, p = stats.spearmanr(sub[var], sub[OUTCOME])
        rows.append({
            "Variable": var, "Scale": "continuous",
            "Test": "Spearman correlation",
            "Statistic": round(rho, 4), "N": n, "p_raw": p,
            "EffectType": "Spearman rho",
            "EffectSize": round(rho, 4),
            "Magnitude": interpret_r(rho),
            "Detail": "monotonic rank association",
        })

    elif scale == "binary":
        groups = [g[OUTCOME].values for _, g in sub.groupby(var)]
        (g1, g2) = groups
        u, p = stats.mannwhitneyu(g1, g2, alternative="two-sided")
        r = rank_biserial(u, len(g1), len(g2))
        rows.append({
            "Variable": var, "Scale": "binary",
            "Test": "Mann-Whitney U",
            "Statistic": round(u, 2), "N": n, "p_raw": p,
            "EffectType": "Rank-biserial r",
            "EffectSize": round(r, 4),
            "Magnitude": interpret_r(r),
            "Detail": group_medians(sub, var),
        })

    elif scale == "multi":
        groups = [g[OUTCOME].values for _, g in sub.groupby(var)]
        h, p = stats.kruskal(*groups)
        e = epsilon_squared(h, n)
        rows.append({
            "Variable": var, "Scale": f"multi ({sub[var].nunique()} grp)",
            "Test": "Kruskal-Wallis H",
            "Statistic": round(h, 3), "N": n, "p_raw": p,
            "EffectType": "Epsilon-squared",
            "EffectSize": round(e, 4),
            "Magnitude": interpret_eps2(e),
            "Detail": group_medians(sub, var),
        })

results = pd.DataFrame(rows)
results

,Variable,Scale,Test,Statistic,N,p_raw,EffectType,EffectSize,Magnitude,Detail
0,V_Subject,binary,Mann-Whitney U,40255.0000,898,7.493733e-27,Rank-biserial r,0.4745,moderate,BTW=1.9; PR=3.0
1,V_Leading_Class,multi (4 grp),Kruskal-Wallis H,18.4210,898,3.601332e-04,Epsilon-squared,0.0205,small,2W=1.9; 4W=2.267; MT_3W=2.067; NMT_3W=1.9
2,Pair,multi (8 grp),Kruskal-Wallis H,157.0780,898,1.321294e-30,Epsilon-squared,0.1751,large,BTW_following_2W=1.633; BTW_following_4W=2.2; ...
3,Target_Speed_km/hr,continuous,Spearman correlation,-0.3901,898,5.164308e-34,Spearman rho,-0.3901,moderate,monotonic rank association
4,Leading_Speed_km/hr,continuous,Spearman correlation,-0.0194,898,5.624495e-01,Spearman rho,-0.0194,negligible,monotonic rank association
5,Speed_Difference,continuous,Spearman correlation,-0.3404,898,8.447454e-26,Spearman rho,-0.3404,moderate,monotonic rank association
6,Off_centeredness,binary,Mann-Whitney U,98455.0000,898,2.947768e-03,Rank-biserial r,-0.1229,weak,False=2.2; True=2.033
7,Occupancy,binary,Mann-Whitney U,73675.5000,898,5.036533e-04,Rank-biserial r,-0.1697,weak,False=2.333; True=2.033
8,Flow_pcu/hr,continuous,Spearman correlation,0.1764,898,1.039142e-07,Spearman rho,0.1764,weak,monotonic rank association
9,Site,binary,Mann-Whitney U,70364.0000,898,7.747131e-05,Rank-biserial r,0.1665,weak,Shahjahanpur=1.9; Tikatuli=2.2


In [6]:
# --- Cell 6: Benjamini-Hochberg FDR correction and formatting ---
def bh_adjust(pvals):
    p = np.asarray(pvals, float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order] * n / (np.arange(1, n + 1))
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adj = np.empty(n)
    adj[order] = np.clip(ranked, 0, 1)
    return adj

results["p_BH"] = bh_adjust(results["p_raw"].values)
results["Significant"] = np.where(results["p_BH"] < ALPHA, "Yes", "No")

# order by absolute effect size, strongest first
results["_abs"] = results["EffectSize"].abs()
results = results.sort_values("_abs", ascending=False).drop(columns="_abs").reset_index(drop=True)

# pretty p-value strings for display / reporting
def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"
results_display = results.copy()
results_display["p_raw"] = results_display["p_raw"].apply(fmt_p)
results_display["p_BH"]  = results_display["p_BH"].apply(fmt_p)

results_display = results_display[[
    "Variable", "Scale", "Test", "Statistic", "N",
    "p_raw", "p_BH", "EffectType", "EffectSize", "Magnitude",
    "Significant", "Detail",
]]
results_display

,Variable,Scale,Test,Statistic,N,p_raw,p_BH,EffectType,EffectSize,Magnitude,Significant,Detail
0,V_Subject,binary,Mann-Whitney U,40255.0000,898,<0.001,<0.001,Rank-biserial r,0.4745,moderate,Yes,BTW=1.9; PR=3.0
1,Target_Speed_km/hr,continuous,Spearman correlation,-0.3901,898,<0.001,<0.001,Spearman rho,-0.3901,moderate,Yes,monotonic rank association
2,Speed_Difference,continuous,Spearman correlation,-0.3404,898,<0.001,<0.001,Spearman rho,-0.3404,moderate,Yes,monotonic rank association
3,Flow_pcu/hr,continuous,Spearman correlation,0.1764,898,<0.001,<0.001,Spearman rho,0.1764,weak,Yes,monotonic rank association
4,Pair,multi (8 grp),Kruskal-Wallis H,157.0780,898,<0.001,<0.001,Epsilon-squared,0.1751,large,Yes,BTW_following_2W=1.633; BTW_following_4W=2.2; ...
5,Occupancy,binary,Mann-Whitney U,73675.5000,898,<0.001,<0.001,Rank-biserial r,-0.1697,weak,Yes,False=2.333; True=2.033
6,Site,binary,Mann-Whitney U,70364.0000,898,<0.001,<0.001,Rank-biserial r,0.1665,weak,Yes,Shahjahanpur=1.9; Tikatuli=2.2
7,Off_centeredness,binary,Mann-Whitney U,98455.0000,898,0.003,0.003,Rank-biserial r,-0.1229,weak,Yes,False=2.2; True=2.033
8,V_Leading_Class,multi (4 grp),Kruskal-Wallis H,18.4210,898,<0.001,<0.001,Epsilon-squared,0.0205,small,Yes,2W=1.9; 4W=2.267; MT_3W=2.067; NMT_3W=1.9
9,Leading_Speed_km/hr,continuous,Spearman correlation,-0.0194,898,0.562,0.562,Spearman rho,-0.0194,negligible,No,monotonic rank association


In [7]:
# --- Cell 7: Save the table to the Tables folder (Excel) ---
out_path = os.path.join(TABLES, "headway_association_nonparametric.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as xl:
    results_display.to_excel(xl, sheet_name="Association_tests", index=False)
    # also keep the numeric (unformatted p) version for later reference
    results.to_excel(xl, sheet_name="Numeric_raw", index=False)
print("Saved:", out_path)

Saved: D:\Headway\Tables\headway_association_nonparametric.xlsx


In [8]:
# --- Cell 8: Effect-size bar chart written into the Excel workbook (no matplotlib) ---
from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference

chart_df = results[["Variable", "EffectSize"]].copy()
chart_df["AbsEffect"] = chart_df["EffectSize"].abs()
chart_df = chart_df.sort_values("AbsEffect", ascending=True)[["Variable", "AbsEffect"]]

wb = load_workbook(out_path)          # out_path comes from Cell 7
if "EffectSize_chart" in wb.sheetnames:
    del wb["EffectSize_chart"]
ws = wb.create_sheet("EffectSize_chart")
ws.append(["Variable", "AbsEffect"])
for _, r in chart_df.iterrows():
    ws.append([r["Variable"], float(r["AbsEffect"])])

chart = BarChart()
chart.type = "bar"
chart.title = "Strength of association with Time Headway"
chart.y_axis.title = "|Effect size|"
chart.x_axis.title = "Variable"
data = Reference(ws, min_col=2, min_row=1, max_row=ws.max_row)
cats = Reference(ws, min_col=1, min_row=2, max_row=ws.max_row)
chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)
chart.height, chart.width = 10, 18
ws.add_chart(chart, "D2")
wb.save(out_path)
print("Chart added to:", out_path)

Chart added to: D:\Headway\Tables\headway_association_nonparametric.xlsx
